# Enhanced S3 to COG Converter with Automatic AWS Authentication

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Support for multiple AWS authentication methods**

Author: Kyle Lesinger (Enhanced version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
import fsspec
from rasterio.warp import calculate_default_transform, reproject, Resampling
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links

[drcs_activations OLD Directory](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/)

[VEDA docs for file naming conventions](https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html)

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [8]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [22]:
EVENT_NAME = '202507_Flood_TX'
#old name
#under drcs_activations
#making two PRODUCT_NAMEs for the two directories mwir files are under
#PRODUCT_NAME = 'wb57/mwir_Jul10/mwir'
#PRODUCT_NAME = 'wb57/mwir'
#PRODUCT_NAME = 'wb57/visible_Jul10/visible'
PRODUCT_NAME = 'wb57/visible'

#under drcs_activations_new
RENAME_PRODUCT = 'WB-57'

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory
DIRECTORY_NEW = f'{DIR_NEW_BASE}/{RENAME_PRODUCT}'

## Initialize AWS S3 Client with automatic credential detection

In [23]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name='nasa-disasters', verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name='nasa-disasters', verbose=True)

# Get all TIF files using the imported function
#keys = get_all_s3_keys(s3_client, 'nasa-disasters', PATH_OLD, ".tif") if s3_client else [] #run the cell first with this line
keys += get_all_s3_keys(s3_client, 'nasa-disasters', PATH_OLD, ".tif") if s3_client else [] #then run this cell with this line   

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
print(len(keys))
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 23740 .tif files in the S3 bucket.
23740


['drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_46_927_mwir_4384.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_48_407_mwir_4385.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_49_427_mwir_4386.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_50_447_mwir_4387.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_51_467_mwir_4388.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_52_478_mwir_4389.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_53_498_mwir_4390.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_54_518_mwir_4391.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0002__191_23_40_55_548_mwir_4392.tif',
 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0003__191_23_42_42_082_mwir_4495.tif',


## Load TIF Files from DRCS Data
### This may assist with diagnosing any issues that occur if no files are found in the code block above

This cell loads the pre-analyzed DRCS activation data from `drcs_activations_tif_files.json` which contains a complete inventory of all .tif files in the NASA Disasters S3 bucket.

The code will:
1. Load the JSON file containing the file inventory
2. Parse the `PATH_OLD` variable to find the corresponding directory
3. Extract all .tif filenames from that directory
4. Store them in `files_to_process` for later use

In [38]:
# # Load the pre-analyzed DRCS TIF files data using imported functions
# # The JSON path is relative to the notebook location
# json_path = Path('../../s3-crawler/drcs_activations_tif_files.json')

# # Load DRCS data
# drcs_data = load_drcs_data(json_path)

# if drcs_data:
#     # Get TIF files from the specified PATH_OLD using the imported function
#     tif_files = get_tif_files_from_path(PATH_OLD, drcs_data, DIR_OLD_BASE)
    
#     if tif_files:
#         print(f"\n📁 Found {len(tif_files)} .tif files in {PATH_OLD}:")
#         print("\nFirst 10 files:")
#         for i, file in enumerate(tif_files[:10], 1):
#             print(f"  {i:2d}. {file}")
#         if len(tif_files) > 10:
#             print(f"  ... and {len(tif_files) - 10} more files")
        
#         # Get files with full paths using the imported function
#         files_to_process = get_files_with_full_paths(PATH_OLD, drcs_data, DIR_OLD_BASE, json_path)
#         print(f"\n✅ Files ready for processing. Stored in 'files_to_process' variable.")
#     else:
#         print(f"\n❌ No files found. Please check the PATH_OLD variable.")
#         files_to_process = []
# else:
#     print(f"\n❌ Could not load DRCS data.")
#     files_to_process = []

# files_to_process

In [39]:
# # Example: List available activation events using the imported function
# print("📂 Available activation events in DRCS data:")
# events = list_available_directories('drcs_activations', drcs_data, json_path)

# # Show first 10 events
# for event in events[:10]:
#     print(f"  - {event}")
# if len(events) > 10:
#     print(f"  ... and {len(events) - 10} more events")

# # Example: List subdirectories for a specific event
# print(f"\n📁 Subdirectories in {EVENT_NAME}:")
# subdirs = list_available_directories(f'drcs_activations/{EVENT_NAME}', drcs_data, json_path)
# for subdir in subdirs:
#     print(f"  - {subdir}")

## Making two directories for visible and mwir

In [25]:
vis = [f for f in keys if "visible" in f]
mwir = [f for f in keys if "mwir" in f]

In [27]:
config_vis = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": [f'{DIR_OLD_BASE}/{EVENT_NAME}/wb57/visible', f'{DIR_OLD_BASE}/{EVENT_NAME}/wb57/visible_Jul10/visible'],
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/visible",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}
config_mwir = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": [f'{DIR_OLD_BASE}/{EVENT_NAME}/wb57/mwir', f'{DIR_OLD_BASE}/{EVENT_NAME}/wb57/mwir_Jul10/mwir'],
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/mwir",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

## Configure bucket and paths (no need to create session manually)

In [28]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [29]:
# Use the function with config_WM
vis_bucket = return_bucket_info(config_vis)
mwir_bucket = return_bucket_info(config_mwir)

Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: ['drcs_activations/202507_Flood_TX/wb57/visible', 'drcs_activations/202507_Flood_TX/wb57/visible_Jul10/visible']
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/WB-57/visible
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: ['drcs_activations/202507_Flood_TX/wb57/mwir', 'drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir']
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/WB-57/mwir


In [9]:
#testing the filename schemes used in the below function
f2 = Path('drcs_activations/202507_Flood_TX/uavsar/colora_11802_25023_006_250709_L090_UNet_predicted_score.tif').stem
f25 = f2.split('_')
print(f25)
f'{EVENT_NAME}_{f2}_202507.tif'

['colora', '11802', '25023', '006', '250709', 'L090', 'UNet', 'predicted', 'score']


'202507_Flood_TX_colora_11802_25023_006_250709_L090_UNet_predicted_score_202507.tif'

In [35]:
def create_cog_filename(f, EVENT_NAME):
    """Create COG filename for mwir and visible files."""
    f2 = Path(f).stem
    cog_filename=f'{EVENT_NAME}_{f2}_202507.tif'
    return cog_filename


# Test functions
print("Testing WM filename:")
test_wm = create_cog_filename('drcs_activations/202507_Flood_TX/wb57/mwir_Jul10/mwir/scan_0054__191_22_43_53_637_mwir_2060.tif', EVENT_NAME)
print(f"  {test_wm}")

Testing WM filename:
  202507_Flood_TX_scan_0054__191_22_43_53_637_mwir_2060_202507.tif


In [31]:
# Define COG profile for rasterio
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

## Define COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with proper CRS and caching.

In [32]:
def convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS.
    
    This function includes:
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    """
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"
    
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)
    
    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    
    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"

    try:
        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(config["raw_data_bucket"], name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        
        # Reproject to EPSG:4326
        print(f"   [REPROJECT] Converting to EPSG:4326...")
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "COG",
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height
                })

                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    for band_idx in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, band_idx),
                            destination=rasterio.band(dst, band_idx),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=dst_crs,
                            resampling=Resampling.nearest,
                            wrapdateline=True
                        )

        # COGify & upload
        print(f"   [COGIFY] Creating COG...")
        ds = rxr.open_rasterio(reproject_filename)
        
        # Handle coordinate naming
        if "y" in ds.dims and "x" in ds.dims:
            ds = ds.rename({"y": "lat", "x": "lon"})
            ds.rio.set_spatial_dims("lon", "lat", inplace=True)
        
        # Smart nodata value handling based on data type
        print(f"   [NODATA] Data type: {ds.dtype}")
        if ds.dtype == 'uint8':
            # For RGB images (uint8), use 0 as nodata (black pixels)
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
        elif ds.dtype == 'uint16':
            # For uint16, use 0 as nodata
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
        else:
            # For float32, int16, int32, etc., use -9999
            nodata_value = -9999
            print(f"   [NODATA] Using nodata value {nodata_value} for {ds.dtype} data")
        
        ds.rio.write_nodata(nodata_value, inplace=True)

        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            tmp_name = tmp.name
            ds.rio.to_raster(tmp_name, **COG_PROFILE)
            
            # Validate COG
            print(f"   [VALIDATE] Checking COG validity...")
            is_valid_cog, validation_details = validate_cog(tmp_name)
            
            if is_valid_cog:
                print(f"   [VALIDATE] ✅ Valid COG")
            else:
                print(f"   [VALIDATE] ⚠️ COG validation warnings")
                critical_errors = [e for e in validation_details['errors'] if 'Invalid driver' in e]
                if critical_errors:
                    raise ValueError(f"Critical COG validation failed")
            
            # Upload to S3
            print(f"   [UPLOAD] Uploading to S3...")
            s3_client.upload_file(
                Filename=tmp_name,
                Bucket=cog_data_bucket,
                Key=s3_key
            )
            print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
            
            # Save locally if specified
            if local_output_dir:
                os.makedirs(local_output_dir, exist_ok=True)
                local_path = os.path.join(local_output_dir, cog_filename)
                import shutil
                shutil.copy(tmp_name, local_path)
            
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)

print("✅ COG conversion function defined with smart nodata handling")

✅ COG conversion function defined with smart nodata handling


In [33]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 14
  - Total size: 1.49 GB

📁 Cached files (first 10):
  - drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_1.tif (233.8 MB)
  - drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_2.tif (233.8 MB)
  - drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Imagery.tif (935.1 MB)
  - drcs_activations/202507_Flood_TX/uavsar/ClassifiedUAVSARI_CopyRas.tif (58.5 MB)
  - drcs_activations/202507_Flood_TX/uavsar/colora_11802_25023_006_250709_L090_UNet_class.tif (1.2 MB)
  - drcs_activations/202507_Flood_TX/uavsar/colora_11802_25023_006_250709_L090_UNet_predicted_score.tif (13.7 MB)
  - drcs_activations/202507_Flood_TX/uavsar/flight25023_mosaic_UNet_class.tif (2.9 MB)
  - drcs_activations/202507_Flood_TX/uavsar/flight25023_mosaic_UNet_class_grayscale.tif (1.3 MB)
  - drcs_activations/202507_Flood_TX/uavsar/guadal_11013_25023_002_250709_L090_UNet_class.tif (0.9 MB)
  - drcs_activations/202507_Flood_TX/uavsar/g

(14, 1595832720)

# Process RGB files

In [12]:
for i in keys:
    head = s3_client.head_object(Bucket=uavsar_bucket['bucket_name'], Key=i)
    print("File size (MB):", head["ContentLength"] / 1024 / 1024)

File size (MB): 233.80956363677979
File size (MB): 233.80956363677979
File size (MB): 935.0884962081909
File size (MB): 58.48189353942871
File size (MB): 1.2417116165161133
File size (MB): 13.682003021240234
File size (MB): 2.865720748901367
File size (MB): 1.301187515258789
File size (MB): 0.8691291809082031
File size (MB): 12.49346923828125
File size (MB): 0.6113147735595703
File size (MB): 15.404109954833984
File size (MB): 1.0707836151123047
File size (MB): 11.175731658935547


In [ ]:
## Process files using batch processing function

"""print("📊 File categorization:")
print(f"  - Water mask files: {len(water_mask)}")
print(f"  - RGB files: {len(rgb)}")
print(f"  - Water mask diff files: {len(water_mask_diff)}")
print(f"  - Total files: {len(keys)}")"""

# Initialize combined results DataFrame
all_files_processed = pd.DataFrame()

config = config_mwir #change here (e.g., the bucket info

# Process water mask files


wm_results = process_file_batch(
    file_list=mwir, #change here )e/g/ l1,l2,l3
    s3_client=s3_client,
    config=config,
    filename_creator_func=create_cog_filename, #change here
    processing_func=convert_to_proper_CRS_and_cogify,
    event_name=EVENT_NAME,
    save_metadata=True,
    save_csv=True,
    verbose=True
)
all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)



# Print overall summary
#print_batch_summary(all_files_processed)

✅ Local output directory ready: output/202507_Flood_TX

[1/11999] Processing: drcs_activations/202507_Flood_TX/wb57/mwir/scan_0008__189_21_51_12_532_mwir_2547.tif
   Output filename: 202507_Flood_TX_scan_0008__189_21_51_12_532_mwir_2547_202507.tif
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326...
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [COGIFY] Creating COG...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/WB-57/mwir/202507_Flood_TX_scan_0008__189_21_51_12_532_mwir_2547_202507.tif
   ✅ Generated and saved COG: 202507_Flood_TX_scan_0008__189_21_51_12_532_mwir_2547_202507.tif

[2/11999] Processing: drcs_activations/202507_Flood_TX/wb57/mwir/scan_0008__189_21_51_14_942_mwir_2548.tif
   Output filename: 

In [1]:
# Display final results
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed


📊 Final Processing Results:


NameError: name 'all_files_processed' is not defined

## Check STATUS
[Disasters Bucket](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/)